# 01.09 Routers & Aggregators — 여러 공급자를 한 코드베이스로

특정 공급자 클래스 대신 **라우터 / 애그리게이터** 를 쓰면 하나의 코드 경로로 수십 개 모델을 교체할 수 있다.

| 래퍼 | 역할 | 대표 사용처 |
|------|------|------------|
| `init_chat_model("openrouter:…")` + OpenRouter base_url | 300+ 모델 **단일 빌링** | 모델 A/B, 폴백 |
| `ChatLiteLLM` | 모든 공급자를 OpenAI 포맷으로 **프록시** | 멀티-클라우드 |
| `ChatTogether` | Together AI 오픈 가중치 | Llama, Qwen 관리형 |
| `ChatFireworks` | Fireworks AI 오픈 가중치 + FireFunction | 고속 open-weight |
| `ChatDeepSeek` | DeepSeek V3 / R1 | reasoning 모델 |
| `ChatXAI` | xAI Grok | Grok-4 reasoning |
| `ChatPerplexity` | Perplexity Sonar | 웹 검색 내장 |

## 학습 목표

- `init_chat_model("openrouter:<model>")` 로 OpenRouter 경유 호출
- 공급자별 래퍼의 import 위치와 환경변수 이름을 매핑
- `ChatLiteLLM` 프록시 패턴으로 멀티-프로바이더 래핑

## 언제 쓰나

- 비용/지연/정확도 기준으로 모델을 **런타임에 교체**해야 하는 라우팅 시스템
- 한 API 키(OpenRouter)로 여러 공급자 빌링을 통합하고 싶을 때
- Perplexity Sonar / xAI Grok / DeepSeek R1 같은 고유 기능이 필요할 때

## 01.09.1 환경 설정

이 노트북은 데모 기본 경로로 **OpenRouter**(`OPENROUTER_API_KEY`)를 사용한다. 다른 공급자 셀들은 각 키가 없으면 해당 부분만 건너뛴다.

In [ ]:
# !pip install -U langchain langchain-openai langchain-community litellm \
#               langchain-together langchain-fireworks langchain-deepseek langchain-xai langchain-perplexity
import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY 필요 (라우터 데모 경로)"

## 01.09.2 OpenRouter — `init_chat_model` 경유

OpenRouter는 OpenAI 호환 포맷이라 `ChatOpenAI`에 `base_url`만 바꿔도 되지만, v1 권장은 `init_chat_model("openai:<model>", base_url=..., api_key=...)` 패턴. 모델 ID는 `<provider>/<model>` 형식(`anthropic/claude-3.5-sonnet`, `google/gemini-2.5-flash` 등).

In [ ]:
from langchain.chat_models import init_chat_model

router = init_chat_model(
    "openai:anthropic/claude-3.5-haiku",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
print(router.invoke("OpenRouter의 핵심 가치는?").content[:200])

## 01.09.3 같은 패턴으로 `ChatOpenAI` 직접 사용

In [ ]:
from langchain_openai import ChatOpenAI

direct = ChatOpenAI(
    model="meta-llama/llama-3.3-70b-instruct",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
print(direct.invoke("한 문장으로 자기소개").content[:200])

## 01.09.4 `ChatLiteLLM` — 모든 공급자를 OpenAI 포맷으로

`litellm`은 100+ 공급자를 OpenAI 포맷으로 프록시하는 어댑터. 멀티-클라우드 라우팅이나 셀프호스팅 게이트웨이에 쓴다. API 키는 각 공급자의 환경변수를 그대로 읽는다(예: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`).

In [ ]:
from langchain_community.chat_models import ChatLiteLLM

# 예: OpenAI GPT를 LiteLLM 경유로 — OPENAI_API_KEY 사용
lite = ChatLiteLLM(model="gpt-4.1-mini")
print(lite.invoke("핑").content[:100])

## 01.09.5 공급자별 래퍼 — import 위치 빠른 맵

아래 셀은 각 래퍼를 **import 만** 하고 클래스를 노출한다. 실제 호출은 해당 키가 있을 때만.

In [ ]:
from langchain_together import ChatTogether
from langchain_fireworks import ChatFireworks
from langchain_deepseek import ChatDeepSeek
from langchain_xai import ChatXAI
from langchain_perplexity import ChatPerplexity

mapping = {
    "ChatTogether":   (ChatTogether, "TOGETHER_API_KEY", "meta-llama/Llama-3.3-70B-Instruct-Turbo"),
    "ChatFireworks":  (ChatFireworks, "FIREWORKS_API_KEY", "accounts/fireworks/models/llama-v3p3-70b-instruct"),
    "ChatDeepSeek":   (ChatDeepSeek, "DEEPSEEK_API_KEY", "deepseek-chat"),
    "ChatXAI":        (ChatXAI, "XAI_API_KEY", "grok-4"),
    "ChatPerplexity": (ChatPerplexity, "PERPLEXITY_API_KEY", "sonar"),
}
for name, (cls, env, model_id) in mapping.items():
    present = "✓" if os.environ.get(env) else " "
    print(f"[{present}] {name:16s} — env: {env:22s} 예시 model: {model_id}")

## 01.09.6 실행 예 — 키가 있을 때만 호출

OpenRouter 경로(이미 키가 있음)로 여러 공급자 모델을 **한 루프**에서 돌려 지연·응답 길이를 비교한다.

In [ ]:
import time

models_to_try = [
    "anthropic/claude-3.5-haiku",
    "google/gemini-2.5-flash",
    "meta-llama/llama-3.3-70b-instruct",
]
for mid in models_to_try:
    m = init_chat_model(
        f"openai:{mid}",
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    t0 = time.time()
    r = m.invoke("LLM 라우터의 가치를 한 문장으로.")
    print(f"{mid:50s} {time.time()-t0:5.2f}s  {r.content[:80]!r}")

## 01.09.7 Tool calling — 라우터 경로에서도 동일

OpenRouter/LiteLLM 어떤 경로든 `bind_tools([...])` 인터페이스는 동일하다. 다만 **선택된 백엔드 모델이 tool calling을 지원**해야 한다.

In [ ]:
from langchain_core.tools import tool

@tool
def square(n: int) -> int:
    """제곱."""
    return n * n

bound = router.bind_tools([square])
print(bound.invoke("9의 제곱은?").tool_calls)

## 체크리스트

- [ ] OpenRouter 는 `init_chat_model("openai:<model>", base_url=..., api_key=...)` 패턴으로 썼다
- [ ] 각 공급자 래퍼의 **패키지 이름 · 환경변수**를 mapping 표로 확인했다
- [ ] 라우터 경로에서도 `bind_tools`·`with_structured_output` 인터페이스가 동일함을 확인했다
- [ ] 백엔드 모델이 tool calling을 지원하지 않으면 래퍼만 바꿔도 동작하지 않는다

## 다음

- `01_openai.ipynb` / `02_anthropic.ipynb` — 공급자 직접 경로
- `11_provider_middleware/` — 공급자별 미들웨어(캐싱, moderation 등)

## 참고

- OpenRouter: https://openrouter.ai/docs
- LiteLLM: https://docs.litellm.ai
- LangChain integrations index: https://docs.langchain.com/oss/python/integrations/chat